In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OCQ_G.xpt"
csv_file = DATA_DIR / "OCQ_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Preserve untouched pandas-decoded source
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 OCQ_G XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OCD150",
    "OCQ180",
    "OCQ210",
    "OCD231",
    "OCD241",
    "OCQ260",
    "OCD270",
    "OCQ275",
    "OCQ380",
    "OCD390G",
    "OCD391",
    "OCD392",
    "OCD395",
    "OCQ510",
    "OCQ520",
    "OCQ530",
    "OCQ540",
    "OCQ550",
    "OCQ560",
    "OCQ570",
    "OCQ580"
]


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "OCQ_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 6175:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 22:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,175 rows x 22 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Never globally replace this value before validating
# exactly where it occurs.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached OCQ_G.xpt:
#
# OCD270 =  62
# OCD395 =  15
# OCQ520 = 187
# OCQ540 = 134
# OCQ560 = 101
# OCQ580 = 181
#
# TOTAL = 680
#

EXPECTED_TINY_COUNTS = {
    "OCD270": 62,
    "OCD395": 15,
    "OCQ520": 187,
    "OCQ540": 134,
    "OCQ560": 101,
    "OCQ580": 181
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated OCQ_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 680:
    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:
    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact 680-value tiny-zero "
    "pattern validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# Do NOT use:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = df[col].eq(TINY_VALUE)

    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:
        raise ValueError(
            f"{col}: expected {expected_count:,} "
            f"tiny values, found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 680:
    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# numeric XPT should contain.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:
    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 9. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# This MUST occur before Int64 conversion.
#
# OCQ_G contains no genuine fractional variables.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:
        fractional_columns.append(
            col
        )


if fractional_columns:
    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in OCQ_G. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):
    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):
    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# OCQ_G contains no genuine decimal variables.
#
# All 22 fields can safely become nullable Int64.
#
# NO .round() is used for conversion.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():
            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely become Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 22 OCQ_G variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 12. HELPER - CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:
        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:
        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 13. CDC CHECK - OCD150
# ============================================================

check_frequency(
    "OCD150",
    {
        1: 3000,
        2: 87,
        3: 297,
        4: 2788,
        7: 2
    },
    1
)


# ============================================================
# 14. CDC CHECK - OCQ180
# ============================================================

s = df["OCQ180"]


if not (
    int(
        s.between(
            1,
            133
        ).sum()
    ) == 2997

    and int(
        s.eq(77777).sum()
    ) == 2

    and int(
        s.eq(99999).sum()
    ) == 1

    and int(
        s.isna().sum()
    ) == 3175
):
    raise ValueError(
        "OCQ180 differs from CDC."
    )


print(
    "PASS: OCQ180 matches CDC."
)


# ============================================================
# 15. CDC CHECK - OCQ210
# ============================================================

check_frequency(
    "OCQ210",
    {
        1: 258,
        2: 632,
        7: 2
    },
    5283
)


# ============================================================
# 16. CURRENT JOB GROUP CODE CHECKS
# ============================================================

if int(
    df["OCD231"]
    .between(
        1,
        22
    )
    .sum()
) != 3047:
    raise ValueError(
        "Unexpected OCD231 valid industry-group count."
    )


if int(
    df["OCD231"]
    .isna()
    .sum()
) != 3128:
    raise ValueError(
        "Unexpected OCD231 missing count."
    )


if int(
    df["OCD241"]
    .between(
        1,
        23
    )
    .sum()
) != 3048:
    raise ValueError(
        "Unexpected OCD241 valid occupation-group count."
    )


if int(
    df["OCD241"]
    .isna()
    .sum()
) != 3127:
    raise ValueError(
        "Unexpected OCD241 missing count."
    )


print(
    "PASS: Current-job industry/occupation "
    "group counts match CDC."
)


# ============================================================
# 17. CDC CHECK - OCQ260
# ============================================================

check_frequency(
    "OCQ260",
    {
        1: 2239,
        2: 115,
        3: 191,
        4: 176,
        5: 325,
        6: 14,
        77: 7,
        99: 20
    },
    3088
)


# ============================================================
# 18. CDC CHECK - OCD270
# ============================================================

#
# Current job duration in months.
#
# 0 = Less than 1 month
#

s = df["OCD270"]


print("\n--- OCD270 ---")

print(
    "Restored zeros:",
    int(
        s.eq(0).sum()
    )
)


if not (
    int(
        s.between(
            1,
            708
        ).sum()
    ) == 3019

    and int(
        s.eq(0).sum()
    ) == 62

    and int(
        s.eq(77777).sum()
    ) == 5

    and int(
        s.eq(99999).sum()
    ) == 1

    and int(
        s.isna().sum()
    ) == 3088
):
    raise ValueError(
        "OCD270 differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: OCD270 matches CDC exactly."
)


# ============================================================
# 19. CDC CHECK - OCQ275
# ============================================================

check_frequency(
    "OCQ275",
    {
        1: 444,
        2: 2640,
        7: 2,
        9: 1
    },
    3088
)


# ============================================================
# 20. CDC CHECK - OCQ380
# ============================================================

check_frequency(
    "OCQ380",
    {
        1: 412,
        2: 529,
        3: 1021,
        4: 214,
        5: 91,
        6: 331,
        7: 190
    },
    3387
)


# ============================================================
# 21. CDC CHECK - OCD390G
# ============================================================

check_frequency(
    "OCD390G",
    {
        1: 3427,
        2: 2120,
        3: 36,
        4: 580,
        7: 10,
        9: 1
    },
    1
)


# ============================================================
# 22. LONGEST JOB GROUP CODE CHECKS
# ============================================================

if int(
    df["OCD391"]
    .between(
        1,
        22
    )
    .sum()
) != 3433:
    raise ValueError(
        "Unexpected OCD391 industry-group count."
    )


if int(
    df["OCD391"]
    .isna()
    .sum()
) != 2742:
    raise ValueError(
        "Unexpected OCD391 missing count."
    )


if int(
    df["OCD392"]
    .between(
        1,
        23
    )
    .sum()
) != 3440:
    raise ValueError(
        "Unexpected OCD392 occupation-group count."
    )


if int(
    df["OCD392"]
    .isna()
    .sum()
) != 2735:
    raise ValueError(
        "Unexpected OCD392 missing count."
    )


print(
    "PASS: Longest-job industry/occupation "
    "group counts match CDC."
)


# ============================================================
# 23. CDC CHECK - OCD395
# ============================================================

#
# Longest job duration in months.
#
# 0 = Less than 1 month
#

s = df["OCD395"]


print("\n--- OCD395 ---")

print(
    "Restored zeros:",
    int(
        s.eq(0).sum()
    )
)


if not (
    int(
        s.between(
            1,
            720
        ).sum()
    ) == 3436

    and int(
        s.eq(0).sum()
    ) == 15

    and int(
        s.eq(77777).sum()
    ) == 2

    and int(
        s.eq(99999).sum()
    ) == 10

    and int(
        s.isna().sum()
    ) == 2712
):
    raise ValueError(
        "OCD395 differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: OCD395 matches CDC exactly."
)


# ============================================================
# 24. CDC CHECK - EXPOSURE YES/NO QUESTIONS
# ============================================================

check_frequency(
    "OCQ510",
    {
        1: 1555,
        2: 3686,
        7: 2,
        9: 9
    },
    923
)


check_frequency(
    "OCQ530",
    {
        1: 1114,
        2: 4132,
        7: 2,
        9: 4
    },
    923
)


check_frequency(
    "OCQ550",
    {
        1: 1195,
        2: 4053,
        7: 2,
        9: 2
    },
    923
)


check_frequency(
    "OCQ570",
    {
        1: 1578,
        2: 3671,
        7: 2,
        9: 1
    },
    923
)


# ============================================================
# 25. CDC CHECK - OCQ520
# ============================================================

#
# Years exposed to mineral dust.
#
# 0 = Less than 1 year
#

s = df["OCQ520"]


if not (
    int(
        s.between(
            0,
            65
        ).sum()
    ) == 1546

    and int(
        s.eq(0).sum()
    ) == 187

    and int(
        s.eq(7777).sum()
    ) == 0

    and int(
        s.eq(9999).sum()
    ) == 9

    and int(
        s.isna().sum()
    ) == 4620
):
    raise ValueError(
        "OCQ520 differs from CDC."
    )


print(
    "PASS: OCQ520 matches CDC; "
    "187 zeros restored."
)


# ============================================================
# 26. CDC CHECK - OCQ540
# ============================================================

#
# Years exposed to organic dust.
#

s = df["OCQ540"]


if not (
    int(
        s.between(
            0,
            65
        ).sum()
    ) == 1112

    and int(
        s.eq(0).sum()
    ) == 134

    and int(
        s.eq(7777).sum()
    ) == 0

    and int(
        s.eq(9999).sum()
    ) == 2

    and int(
        s.isna().sum()
    ) == 5061
):
    raise ValueError(
        "OCQ540 differs from CDC."
    )


print(
    "PASS: OCQ540 matches CDC; "
    "134 zeros restored."
)


# ============================================================
# 27. CDC CHECK - OCQ560
# ============================================================

#
# Years exposed to exhaust fumes.
#

s = df["OCQ560"]


if not (
    int(
        s.between(
            0,
            55
        ).sum()
    ) == 1193

    and int(
        s.eq(0).sum()
    ) == 101

    and int(
        s.eq(7777).sum()
    ) == 0

    and int(
        s.eq(9999).sum()
    ) == 2

    and int(
        s.isna().sum()
    ) == 4980
):
    raise ValueError(
        "OCQ560 differs from CDC."
    )


print(
    "PASS: OCQ560 matches CDC; "
    "101 zeros restored."
)


# ============================================================
# 28. CDC CHECK - OCQ580
# ============================================================

#
# Years exposed to other gases/vapors/fumes.
#

s = df["OCQ580"]


if not (
    int(
        s.between(
            0,
            60
        ).sum()
    ) == 1577

    and int(
        s.eq(0).sum()
    ) == 181

    and int(
        s.eq(7777).sum()
    ) == 0

    and int(
        s.eq(9999).sum()
    ) == 1

    and int(
        s.isna().sum()
    ) == 4597
):
    raise ValueError(
        "OCQ580 differs from CDC."
    )


print(
    "PASS: OCQ580 matches CDC; "
    "181 zeros restored."
)


# ============================================================
# 29. EXPOSURE ROUTING CHECKS - REVISED
# ============================================================

#
# Duration should be populated only when the
# corresponding exposure question is Yes (= 1).
#
# IMPORTANT:
#
# These columns are pandas nullable Int64.
#
# df[col].eq(1) can therefore return pd.NA
# wherever the source field is missing.
#
# np.array_equal() cannot safely compare pd.NA.
#
# We explicitly convert missing Boolean results
# to False ONLY in this temporary validation mask.
#
# THIS DOES NOT MODIFY THE DATAFRAME.
#

EXPOSURE_ROUTING = {
    "OCQ520": "OCQ510",
    "OCQ540": "OCQ530",
    "OCQ560": "OCQ550",
    "OCQ580": "OCQ570"
}


print("\n--- EXPOSURE ROUTING CHECK ---")


for duration_col, exposure_col in EXPOSURE_ROUTING.items():

    #
    # True if duration value exists.
    #
    duration_nonmissing = (
        df[duration_col]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )


    #
    # True only when exposure question == 1.
    #
    # Missing responses become False in this
    # temporary Boolean mask only.
    #
    exposure_yes = (
        df[exposure_col]
        .eq(1)
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )


    mismatch_mask = (
        duration_nonmissing
        !=
        exposure_yes
    )


    mismatch_count = int(
        mismatch_mask.sum()
    )


    print(
        f"{exposure_col} -> {duration_col}: "
        f"{mismatch_count:,} routing mismatches"
    )


    if mismatch_count != 0:

        problem_rows = (
            df.loc[
                mismatch_mask,
                [
                    "SEQN",
                    exposure_col,
                    duration_col
                ]
            ]
            .head(20)
        )


        print(
            "\nExample routing mismatches:"
        )


        print(
            problem_rows.to_string(
                index=False
            )
        )


        raise ValueError(
            f"{duration_col} routing does not "
            f"exactly match {exposure_col}=1."
        )


    print(
        f"PASS: {duration_col} routing "
        f"matches {exposure_col}=1."
    )


print(
    "\nPASS: All four occupational exposure "
    "routing checks passed."
)


# ============================================================
# 30. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original attached XPT contained no ordinary zeros.
#
# Therefore all final zeros should occur only in
# the six validated fields.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "Final zero pattern differs from "
        "validated XPORT pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {final_zero_counts}"
    )


print(
    "PASS: Final zero pattern exactly matches "
    "the six validated corrections."
)


# ============================================================
# 31. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Only permitted numerical changes:
#
# OCD270 =  62 tiny -> 0
# OCD395 =  15 tiny -> 0
# OCQ520 = 187 tiny -> 0
# OCQ540 = 134 tiny -> 0
# OCQ560 = 101 tiny -> 0
# OCQ580 = 181 tiny -> 0
#
# Nothing else may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:
        fidelity_failures.append(
            col
        )


if fidelity_failures:
    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 22 variables match the "
    "expected corrected XPT exactly."
)


# ============================================================
# 32. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO rounding.
# NO float_format.
#
# Although OCQ_G has no genuine decimal variables,
# this keeps the same safe export pattern used across
# all NHANES files.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 33. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:
    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):
    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 34. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:
        csv_failures.append(
            col
        )


if csv_failures:
    raise ValueError(
        "Corrected XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 22 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 35. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:
    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 36. READ RAW CSV AS TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 37. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:
    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 38. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# OCQ_G has no genuine decimal variables.
#
# Therefore every populated value may safely be checked
# for unnecessary ".0".
#
# DO NOT use this global rule for Dietary/Lab/Exam files
# containing true decimal measurements.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:
        dot_zero_errors[col] = count


if dot_zero_errors:
    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting "
    "in any OCQ_G variable."
)


# ============================================================
# 39. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    count = int(
        values.eq(0).sum()
    )


    if count > 0:
        csv_zero_counts[col] = count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "Final CSV zero pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly "
    "matches validated corrections."
)


# ============================================================
# 40. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:
    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 41. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):
    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):
    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 42. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "OCQ_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)

print(
    "OCD270 zero restorations:",
    EXPECTED_TINY_COUNTS["OCD270"]
)

print(
    "OCD395 zero restorations:",
    EXPECTED_TINY_COUNTS["OCD395"]
)

print(
    "OCQ520 zero restorations:",
    EXPECTED_TINY_COUNTS["OCQ520"]
)

print(
    "OCQ540 zero restorations:",
    EXPECTED_TINY_COUNTS["OCQ540"]
)

print(
    "OCQ560 zero restorations:",
    EXPECTED_TINY_COUNTS["OCQ560"]
)

print(
    "OCQ580 zero restorations:",
    EXPECTED_TINY_COUNTS["OCQ580"]
)

print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)

print(
    "Genuine decimal variables: NONE"
)

print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary .0: "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 OCQ_G XPT -> CSV
Rows: 6,175
Columns: 22
PASS: Structure = 6,175 rows x 22 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 680

Tiny values by variable:
OCQ520    187
OCQ580    181
OCQ540    134
OCQ560    101
OCD270     62
OCD395     15

PASS: Exact 680-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
OCD270: 62 restored to 0
OCD395: 15 restored to 0
OCQ520: 187 restored to 0
OCQ540: 134 restored to 0
OCQ560: 101 restored to 0
OCQ580: 181 restored to 0

PASS: 680 validated zero values restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
OCD150: 0 fractional observations
OCQ180: 0 fractional observations
OCQ210: 0 fractional observations
OCD231: 0 fractional observations
OCD241: 0 fractional observations
OCQ260: 0 fractional observations
OCD270: 0 fractional observations
OCQ275: 0 fractional observations
O